<a href="https://colab.research.google.com/github/chankurjawla/dashboard/blob/Dev/MF_Allotment_email_reader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
!pip install imapclient

In [134]:
from imapclient import IMAPClient
from email import message_from_bytes
import pandas as pd
import re
from datetime import datetime
from bs4 import BeautifulSoup
from io import StringIO
import os

HOST = "imap.gmail.com"
#USERNAME = "ch.ankurjawla@gmail.com"
#PASSWORD = "lupx encd vahn kqym"
LABEL = "Allotment"
file_name = 'equity_allotment.csv'

In [137]:
# Redefine fetch_and_extract_tables with label removal functionality
def fetch_and_extract_tables(host, username, password, label):
    extracted_data = []
    try:
        with IMAPClient(host) as client:
            client.login(username, password)
            client.select_folder(label)
            messages = client.search('ALL')

            for msg_id in messages:
                raw_message = client.fetch(msg_id, ['RFC822'])
                email_message = message_from_bytes(raw_message[msg_id][b'RFC822'])

                email_subject = email_message['Subject'] if 'Subject' in email_message else 'No Subject'

                html_content = None
                for part in email_message.walk():
                    if part.get_content_type() == 'text/html':
                        charset = part.get_content_charset()
                        html_content = part.get_payload(decode=True).decode(charset or 'utf-8', errors='replace')
                        break

                if html_content:
                    soup = BeautifulSoup(html_content, 'html.parser')
                    tables = soup.find_all('table')

                    email_dataframes = []
                    for table_idx, table in enumerate(tables):
                        try:
                            dfs = pd.read_html(StringIO(str(table)))
                            if dfs:
                                for df in dfs:
                                    email_dataframes.append(df)
                        except Exception as e:
                            print(f"Warning: Could not parse table {table_idx} from email (Subject: '{email_subject}'): {e}")

                    if email_dataframes:
                        extracted_data.append({
                            'email_subject': email_subject,
                            'dataframes': email_dataframes
                        })
                        # --- Add label removal here ---
                        try:
                            # IMAPClient uses flags for labels. A custom label is a flag.
                            client.remove_flags(msg_id, [label], silent=True)
                            print(f"Removed label '{label}' from email (Subject: '{email_subject}', ID: {msg_id})")
                        except Exception as e:
                            print(f"Error removing label from email (Subject: '{email_subject}', ID: {msg_id}): {e}")

                else:
                    print(f"Info: No HTML content found in email (Subject: '{email_subject}')")

    except Exception as e:
        print(f"An error occurred during email fetching or processing: {e}")

    return extracted_data


In [138]:
def extract_table(USERNAME,PASSWORD):
    extracted_tables_data = fetch_and_extract_tables(HOST, USERNAME, PASSWORD, LABEL)

    df_allotment = extracted_tables_data[0]['dataframes'][0]

    pattern = r'([₹$]?[\d,]+\.?\d*)\s*(.*)'

    df_allotment[['Amount_str', 'unit']] = df_allotment['Amount'].astype(str).str.extract(pattern)

    df_allotment['Amount'] = df_allotment['Amount_str'].str.replace(r'[₹$]', '', regex=True).str.replace(',', '').astype(float)
    df_allotment['unit'] = df_allotment['unit'].str.replace(' units', '', regex=False).astype(float)
    # Drop the intermediate 'Amount_str' column
    df_allotment = df_allotment.drop(columns=['Amount_str'])
    df_allotment = df_allotment.rename(columns={'Fund': 'Ticker'})
    df_allotment['Ticker'] = df_allotment['Ticker'].str.split(' Folio').str[0]
    df_allotment = df_allotment.groupby('Ticker').agg({'Amount': 'mean', 'unit': 'sum'}).reset_index() # in case there are more than one folio for same mutual fund

    if USERNAME == "ch.ankurjawla@gmail.com":
        df_allotment["Owner"] = "Ankur"
    else:
        df_allotment['Owner'] = "Gulu"

    email_subject = extracted_tables_data[0]['email_subject']
    date_pattern = r'\d{2}-\d{2}-\d{4}'
    match = re.search(date_pattern, email_subject)

    if match:
        date_str = match.group(0)
        df_allotment['Date'] = pd.to_datetime(date_str, format='%d-%m-%Y')
    else:
        df_allotment['Date'] = pd.NaT
        print("No date found in email subject.")

    return df_allotment

In [139]:
def push_to_csv(USERNAME,PASSWORD):
    df_allotment = extract_table(USERNAME,PASSWORD)

    if os.path.exists(file_name):
        # If file exists, append without writing header
        df_allotment.to_csv(file_name, mode='a', header=False, index=False)
        print(f"Data appended to {file_name}")
    elif 'df_allotment' in locals() and not df_allotment.empty:
        # If file does not exist, create it and write with header
        df_allotment.to_csv(file_name, mode='w', header=True, index=False)
        print(f"Data written to {file_name}")
    else:
        print("df_allotment does not exist or is empty. No data to write.")

In [141]:
push_to_csv(USERNAME,PASSWORD)

Removed label 'Allotment' from email (Subject: 'Coin by Zerodha - Allotment report - 09-03-2026', ID: 1)
Data appended to equity_allotment.csv


## fetch mutual funds' NAV

In [142]:
mf_json = {
    "NAVI NIFTY BANK INDEX FUND - DIRECT PLAN GROWTH": "0P0001OAEY.BO",
    "TATA DIGITAL INDIA FUND DIRECT PLAN GROWTH":"0P0001784G.BO",
    "SBI FOCUSED FUND DIRECT GROWTH":"0P0000XVJT.BO",
    "MIRAE ASSET LARGE CAP FUND - DIRECT PLAN - GROWTH":"0P0000XVA0.BO",
    "NIPPON INDIA SMALL CAP FUND DIRECT PLAN GROWTH PLAN GROWTH":"0P0000XVFY.BO"
}

mfjson = pd.DataFrame(list(mf_json.items()), columns=['Fund Name', 'Ticker'])
mfjson.to_json('mutual_funds.json', orient='records')


In [122]:
pd.read_json('mutual_funds.json')

,Fund Name,Ticker
0,NAVI NIFTY BANK INDEX FUND - DIRECT PLAN GROWTH,0P0001OAEY.BO
1,TATA DIGITAL INDIA FUND DIRECT PLAN GROWTH,0P0001784G.BO
2,SBI FOCUSED FUND DIRECT GROWTH,0P0000XVJT.BO


In [144]:
import yfinance as yf
import pandas as pd
from datetime import datetime

file_name = 'equity_price.csv'
# Load the updated mutual funds list from the JSON file
updated_mfjson = pd.read_json('mutual_funds.json')
ticker_symbols = updated_mfjson['Ticker'].tolist()

current_day = datetime.now().strftime('%Y-%m-%d')
final_df = pd.DataFrame()
for ticker_symbol in ticker_symbols:
    print(f"Fetching data for {ticker_symbol}...")
    ticker = yf.Ticker(ticker_symbol)
    # Start date set to a reasonable historical period, or adjust as needed
    history = ticker.history(start="2026-01-01", end=current_day)
    if not history.empty:
        history.rename(columns={'Close': ticker_symbol}, inplace=True)
        if final_df.empty:
            final_df = history[[ticker_symbol]]
        else:
            final_df = pd.concat([final_df, history[[ticker_symbol]]], axis=1)
    else:
        print(f"No historical data found for {ticker_symbol} within the specified date range.")

if os.path.exists(file_name):
    # If file exists, append without writing header
        final_df.to_csv(file_name, mode='a', header=False, index=False)
        print(f"Data appended to {file_name}")
    elif 'final_df' in locals() and not final_df.empty:
        # If file does not exist, create it and write with header
        final_df.to_csv(file_name, mode='w', header=True, index=False)
        print(f"Data written to {file_name}")
    else:
        print("df_allotment does not exist or is empty. No data to writ")


Fetching data for 0P0001OAEY.BO...
Data written to equity_price.csv
Fetching data for 0P0001784G.BO...
Data appended to equity_price.csv
Fetching data for 0P0000XVJT.BO...
Data appended to equity_price.csv
Fetching data for 0P0000XVA0.BO...
Data appended to equity_price.csv
Fetching data for 0P0000XVFY.BO...
Data appended to equity_price.csv
